In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
# run_pattern = "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
run_pattern = "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 9 runs: ['total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_1204_180157', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_1304_173343', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_1204_192553', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_1204_204954', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_1204_221356', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_1204_233758', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_1304_010158', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_1304_022602', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_1304_035010']


In [4]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'camels_01411300': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.75 1.99 2.85 ... 0.46 0.55
       streamflow_sim  (date, time_step) float32 15kB 1.138 1.282 ... 0.6935 0.8208}},
 'camels_01487000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.43 1.88 2.14 ... 1.47 1.58
       streamflow_sim  (date, time_step) float32 15kB 1.986 2.257 ... 0.7342 0.956}},
 'camels_01491000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordina

In [5]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}
for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    all_metrics[basin_id] = calculate_metrics(
        obs=xr_ds['streamflow_obs'], #['QObs_mm_d_obs'],
        sim=xr_ds['streamflow_sim'], #['QObs_mm_d_sim'],
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'

df_metrics

,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
camels_01411300,0.424577,0.980015,0.989957,0.398556,0.518208,0.717077,0.777368,-0.223769,-43.678432,-1.953835e+01,6.012143e+00,0.250000,0.471698,51.875221
camels_01487000,0.636558,0.498962,0.706373,0.775476,0.892763,0.802937,0.991194,-0.008995,-9.784943,-8.151369e+00,-7.914616e+01,0.333333,0.476190,43.200039
camels_01491000,0.505231,2.175454,1.474942,0.487738,0.566837,0.729544,1.040444,0.023031,-42.938179,-2.531106e+01,5.346240e+01,0.444444,0.549020,69.896606
camels_01644000,0.325167,2.591987,1.609965,0.635494,1.067334,0.690465,1.180331,0.097704,12.345016,-1.777965e+01,9.033395e+01,0.636364,0.622642,65.708740
camels_01664000,0.439399,2.530969,1.590902,0.695905,0.940606,0.704139,0.962412,-0.022274,-4.175384,-3.524161e+00,6.938376e+01,0.238095,0.559322,58.180531
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
camels_08200000,0.209558,4.675666,2.162329,0.085154,0.329334,0.490163,0.643327,-0.069538,-52.584602,-2.851278e+01,6.346478e+01,0.500000,0.459459,92.934814
camels_08202700,-0.050329,1.670363,1.292425,-2.828800,0.475684,0.205419,4.708563,0.139564,43.892803,1.337636e+08,-1.106849e+12,0.500000,0.142857,74.572426
camels_09484600,-0.012336,0.003372,0.058071,0.056343,0.657059,0.327619,1.566378,0.105549,-32.057590,1.916784e+08,-2.576778e+11,0.769231,0.627907,69.688354


In [6]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(f"./ensemble_metrics/{save_name}.csv")

In [7]:
df_metrics.median()

NSE              0.289797
MSE              4.836532
RMSE             2.199211
KGE              0.471666
Alpha-NSE        0.760924
Pearson-r        0.614236
Beta-KGE         0.990495
Beta-NSE        -0.005433
FHV            -20.409995
FMS            -19.546385
FLV              8.753955
Peak-Timing      0.473684
Missed-Peaks     0.552715
Peak-MAPE       64.180908
dtype: float64